In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import torch
import torch.nn as nn
from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

1) Design Model (input, output size, forward pass)
2) Construct loss and optimizer
3) Training loop
	- forward pass: compute prediction
	- backward pass: gradients
	- update weights


In [3]:
bc = datasets.load_breast_cancer()

X, y = bc.data, bc.target

In [4]:
n_samples, n_features = X.shape

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X,y, train_size=0.8, random_state=1903)

In [6]:
# scale features

sc = StandardScaler()

X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [7]:
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32))
y_test = torch.from_numpy(y_test.astype(np.float32))

In [8]:
y_train = y_train.view(y_train.shape[0], 1)
y_test = y_test.view(y_test.shape[0], 1)

In [9]:
# model

class LogisticRegression(nn.Module):
    def __init__(self, n_input_features):
        super(LogisticRegression, self).__init__()
        self.linear = nn.Linear(n_input_features, 1)

    def forward(self, x):
        y_pred = torch.sigmoid(self.linear(x))
        return y_pred

model = LogisticRegression(n_features)

In [10]:
learning_rate = 0.01

criterion = nn.BCELoss() # binary cross entropy
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [11]:
n_epochs = 500

for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)

    loss.backward()

    optimizer.step()

    optimizer.zero_grad()

    print(f"epoch {epoch + 1}: loss = {loss.item():.4f}")

epoch 1: loss = 0.8764
epoch 2: loss = 0.8452
epoch 3: loss = 0.8157
epoch 4: loss = 0.7878
epoch 5: loss = 0.7614
epoch 6: loss = 0.7366
epoch 7: loss = 0.7133
epoch 8: loss = 0.6913
epoch 9: loss = 0.6706
epoch 10: loss = 0.6512
epoch 11: loss = 0.6329
epoch 12: loss = 0.6157
epoch 13: loss = 0.5996
epoch 14: loss = 0.5843
epoch 15: loss = 0.5700
epoch 16: loss = 0.5564
epoch 17: loss = 0.5437
epoch 18: loss = 0.5316
epoch 19: loss = 0.5202
epoch 20: loss = 0.5095
epoch 21: loss = 0.4993
epoch 22: loss = 0.4896
epoch 23: loss = 0.4804
epoch 24: loss = 0.4717
epoch 25: loss = 0.4634
epoch 26: loss = 0.4555
epoch 27: loss = 0.4480
epoch 28: loss = 0.4409
epoch 29: loss = 0.4341
epoch 30: loss = 0.4275
epoch 31: loss = 0.4213
epoch 32: loss = 0.4153
epoch 33: loss = 0.4096
epoch 34: loss = 0.4041
epoch 35: loss = 0.3988
epoch 36: loss = 0.3938
epoch 37: loss = 0.3889
epoch 38: loss = 0.3843
epoch 39: loss = 0.3798
epoch 40: loss = 0.3754
epoch 41: loss = 0.3712
epoch 42: loss = 0.3672
e

In [12]:
# no grad ve grad mantığı aklıma gelen en sade mantıkla train testte bariz beliriyor
# backward grad falan varsa işin içinde öğrenme var bu yüzden testte grad kullanmıyoruz.
with torch.no_grad():
    y_pred = model(X_test)
    y_pred_cls = y_pred.round()

    acc = y_pred_cls.eq(y_test).sum() / float(y_test.shape[0])
    print("accuracy = ",acc)

accuracy =  tensor(0.9649)
